In [0]:
import pandas as pd
import numpy as np
import time

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, rand, row_number, ceil, count, date_format, mean
from pyspark.sql.functions import collect_list, struct, array, lit
from pyspark.sql.functions import aggregate, expr, sum as spark_sum, sqrt
from pyspark.sql.window import Window

movielens_base = "/Volumes/workspace/default/movielens/"


In [0]:
# rating: 
rating_schema = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("movie_id", IntegerType(), nullable=False),
    StructField("rating", DoubleType(), nullable=True),
    StructField("rating_timestamp", TimestampType(), nullable=True)
])

rating = spark.read.csv(movielens_base + "rating.csv", header=True, schema=rating_schema, mode="FAILFAST", timestampFormat="yyyy-MM-dd HH:mm:ss")

display(rating.withColumn("rating_timestamp", date_format("rating_timestamp", "yyyy-MM-dd HH:mm:ss")).limit(5))



In [0]:

# Serverless compute in Databricks doesn't support cache()/localCheckpoint().
# The data is small, so collecting and recreating the DataFrame breaks the lineage.
# For large data, P and Q would be written to a Delta table each iteration instead.
def materialize(df):
    return spark.createDataFrame(df.toPandas(), schema=df.schema)
    
def update_user_factors(pdf):
    user_id = int(pdf["user_id"].iloc[0])
    r_u = pdf["rating"].to_numpy(dtype=float)

    q_u = np.array(pdf["movie_factors"].tolist(), dtype=float)

    p_u = np.linalg.solve(
        q_u.T @ q_u + lambda_ * I,
        q_u.T @ r_u
    )

    return pd.DataFrame({
        "user_id": [user_id],
        "factors": [p_u.tolist()]
    })


def update_movie_factors(pdf):
    movie_id = int(pdf["movie_id"].iloc[0])
    r_i = pdf["rating"].to_numpy(dtype=float)

    p_i = np.array(pdf["user_factors"].tolist(), dtype=float)

    q_i = np.linalg.solve(
        p_i.T @ p_i + lambda_ * I,
        p_i.T @ r_i
    )

    return pd.DataFrame({
        "movie_id": [movie_id],
        "factors": [q_i.tolist()]
    })


# Configuration
rng = np.random.default_rng(seed=42)

k = 5
lambda_ = 4
I = np.eye(k)
delta_E_min = 0.1
max_loop_count = 2000
training_fraction = 0.8

max_user_cnt = 100
spark.conf.set("spark.sql.shuffle.partitions", 8) # for a small number of users 


# Select the first 100 users from MovieLens data.
# Keep the original user_id and movie_id values.
rating_100_users = (
    rating
    .filter(col("user_id").between(1, max_user_cnt))
)

#rating_100_users.select("user_id").distinct().orderBy("user_id").show(100, truncate=False)

# Randomly rank ratings separately for each user.
w = Window.partitionBy("user_id").orderBy(rand(seed=42))

rating_100_users = rating_100_users.withColumn(
    "rating_rank",
    row_number().over(w)
)

# Count ratings per user.
w_user = Window.partitionBy("user_id")

rating_100_users = rating_100_users.withColumn(
    "user_rating_count",
    count("*").over(w_user)
)

# Assign the first training_fraction of ratings to training.
rating_100_users = rating_100_users.withColumn(
    "validation",
    col("rating_rank") >
    ceil(col("user_rating_count") * training_fraction)
)

# Evaluate the random split once, so training and validation are each other's complement
rating_100_users = materialize(rating_100_users)

# Create training and validation datasets.
rating_100_users_train = rating_100_users.filter(~col("validation"))
rating_100_users_val = rating_100_users.filter(col("validation"))

rating_100_users_spark_train = rating_100_users_train.select(
    "user_id", "movie_id", "rating"
)

rating_100_users_spark_val = rating_100_users_val.select(
    "user_id", "movie_id", "rating"
)

rating_100_users_spark_train = materialize(
    rating_100_users_train.select("user_id", "movie_id", "rating")
)
rating_100_users_spark_train.count()   # materialize it now

rating_100_users_spark_val = materialize(
    rating_100_users_val.select("user_id", "movie_id", "rating")
)
rating_100_users_spark_val.count()   # materialize it now

# Measure execution time.
start = time.perf_counter()

user_cnt = rating_100_users.select("user_id").distinct().count()
movie_cnt = rating_100_users.select("movie_id").distinct().count()

print(f"user_cnt = {user_cnt}")
print(f"movie_cnt = {movie_cnt}")

# Initialise movie factors using the original movie IDs.
Q = (
    rating_100_users_spark_train
    .select("movie_id")
    .distinct()
    .withColumn(
        "factors",
        array(*[
            rand(seed=42 + j) * 0.2 - 0.1
            for j in range(k)
        ])
    )
)

# Initialise user factors using the original user IDs.
P = (
    rating_100_users_spark_train
    .select("user_id")
    .distinct()
    .withColumn(
        "factors",
        array(*[lit(0.0) for _ in range(k)])
    )
)


# initialze variable for loop
E = 1000
E_cur = 900
loop_count = 0
error_history = []

while (
    abs(E - E_cur) > delta_E_min
    and loop_count < max_loop_count
):
    E = E_cur

    # Join training ratings with current movie factors.
    ratings_with_movie_factors = (
        rating_100_users_spark_train
        .join(Q, on="movie_id", how="inner")
        .withColumnRenamed("factors", "movie_factors")
    )

    # Update user factors using training ratings only.
    P = materialize(
        ratings_with_movie_factors
        .groupBy("user_id")
        .applyInPandas(
            update_user_factors,
            schema="user_id long, factors array<double>"
        )
    )

    # Join training ratings with current user factors.
    ratings_with_user_factors = (
        rating_100_users_spark_train
        .join(P, on="user_id", how="inner")
        .withColumnRenamed("factors", "user_factors")
    )

    # Update movie factors using training ratings only.
    Q = materialize(
        ratings_with_user_factors
        .groupBy("movie_id")
        .applyInPandas(
            update_movie_factors,
            schema="movie_id long, factors array<double>"
        )
    )

    # Join training ratings with current user and movie factors.
    ratings_with_factors = (
        rating_100_users_spark_train
        .join(
            P.withColumnRenamed("factors", "user_factors"),
            on="user_id"
        )
        .join(
            Q.withColumnRenamed("factors", "movie_factors"),
            on="movie_id"
        )
    )

    # Calculate squared prediction errors.
    ratings_with_errors = ratings_with_factors.withColumn(
        "squared_error",
        expr("""
            pow(
                rating - aggregate(
                    zip_with(
                        user_factors,
                        movie_factors,
                        (p, q) -> p * q
                    ),
                    cast(0.0 as double),
                    (acc, x) -> acc + x
                ),
                2
            )
        """)
    )

    # Sum squared errors.
    squared_error_sum = ratings_with_errors.agg(
        spark_sum("squared_error").alias("sse")
    ).first()["sse"]

    # Sum squared user factors.
    user_norm_sum = P.select(
        spark_sum(
            expr("""
                aggregate(
                    factors,
                    cast(0.0 as double),
                    (acc, x) -> acc + x*x
                )
            """)
        ).alias("norm")
    ).first()["norm"]

    # Sum squared movie factors.
    movie_norm_sum = Q.select(
        spark_sum(
            expr("""
                aggregate(
                    factors,
                    cast(0.0 as double),
                    (acc, x) -> acc + x*x
                )
            """)
        ).alias("norm")
    ).first()["norm"]

    # Regularised ALS objective.
    E_cur = squared_error_sum + lambda_ * (
        user_norm_sum + movie_norm_sum
    )

    loop_count += 1
    print(f"Error = {E_cur}")
    error_history.append((loop_count, float(E_cur)))


print(f"Total error = {E_cur}. Loop count = {loop_count}")

print(
    f"Mean rating training data = "
    f"{rating_100_users_spark_train.select(mean('rating')).first()[0]:.4f}"
)


# Predict ratings for the training dataset.
training_predictions = (
    rating_100_users_spark_train
    .join(
        P.withColumnRenamed("factors", "user_factors"),
        on="user_id"
    )
    .join(
        Q.withColumnRenamed("factors", "movie_factors"),
        on="movie_id"
    )
    .withColumn(
        "prediction",
        expr("""
            aggregate(
                zip_with(
                    user_factors,
                    movie_factors,
                    (p, q) -> p * q
                ),
                cast(0.0 as double),
                (acc, x) -> acc + x
            )
        """)
    )
)

# Calculate training RMSE.
rmse = training_predictions.select(
    sqrt(
        mean(pow(col("rating") - col("prediction"), 2))
    ).alias("rmse")
).first()["rmse"]

print(f"Training data RMSE = {rmse:.4f}")

elapsed = time.perf_counter() - start
print(f"ALS training time: {elapsed:.2f} seconds")


In [0]:

## Test prediction on validation dataset
from pyspark.sql.functions import (
    col, mean, sqrt, pow, abs as spark_abs,
    count, expr, min as spark_min, max as spark_max
)

# Count training ratings per user and per movie
user_train_counts = (
    rating_100_users_spark_train
    .groupBy("user_id")
    .agg(count("*").alias("user_train_count"))
)

movie_train_counts = (
    rating_100_users_spark_train
    .groupBy("movie_id")
    .agg(count("*").alias("movie_train_count"))
)

# Predict ratings on validation data
validation_predictions = (
    rating_100_users_spark_val
    .join(
        P.withColumnRenamed("factors", "user_factors"),
        on="user_id", how="left"
    )
    .join(
        Q.withColumnRenamed("factors", "movie_factors"),
        on="movie_id", how="left"
    )
    .withColumn(
        "prediction",
        expr("""
            aggregate(
                zip_with(user_factors, movie_factors, (p, q) -> p * q),
                cast(0.0 as double),
                (acc, x) -> acc + x
            )
        """)
    )
    .withColumn(
        "error",
        spark_abs(col("rating") - col("prediction"))
    )
    .join(user_train_counts, on="user_id", how="left")
    .join(movie_train_counts, on="movie_id", how="left")
)

# Validation summary
summary = validation_predictions.agg(
    mean("rating").alias("mean_actual"),
    mean("prediction").alias("mean_predicted"),
    sqrt(mean(pow(col("rating") - col("prediction"), 2))).alias("rmse"),
    spark_min("prediction").alias("min_predicted"),
    spark_max("prediction").alias("max_predicted")
).first()

print(f"Mean rating validation data = {summary['mean_actual']:.4f}")
print(f"Validation RMSE = {summary['rmse']:.4f}")
print(f"Mean predicted rating = {summary['mean_predicted']:.4f}")
print(f"Min predicted rating  = {summary['min_predicted']:.4f}")
print(f"Max predicted rating  = {summary['max_predicted']:.4f}")

# Some movies only appear in the validation set and have no training ratings.
# Spark: such movies have no row in Q, so the left join gives movie_factors = null
#        and prediction = null. Spark's mean() ignores nulls, so these ratings are
#        excluded from the validation RMSE.
n_total = validation_predictions.count()
n_missing = validation_predictions.filter(col("prediction").isNull()).count()
print(f"Validation ratings without prediction = {n_missing} of {n_total}")

# Show the 10 largest absolute errors
worst = (
    validation_predictions
    .orderBy(col("error").desc())
    .limit(10)
    .collect()
)

for row in worst:
    print(
        f"user={row['user_id']} "
        f"(training ratings={row['user_train_count']}), "
        f"movie={row['movie_id']} "
        f"(training ratings={row['movie_train_count']}), "
        f"actual={row['rating']:.1f}, "
        f"predicted={row['prediction']:.2f}, "
        f"error={row['error']:.2f}"
    )

    print(
        f"    ||P[u]||={sum(x*x for x in row['user_factors'])**0.5:.3f}, "
        f"||Q[i]||={sum(x*x for x in row['movie_factors'])**0.5:.3f}, "
        f"movie training ratings={row['movie_train_count']}"
    )


In [0]:
# save tables to make them available for visualisation
tbl = "workspace.default"

(spark.createDataFrame(error_history, ["iteration", "objective"])
    .withColumn("implementation", lit("spark"))
    .write.mode("overwrite")
    .option("replaceWhere", "implementation = 'spark'")
    .saveAsTable(f"{tbl}.als_convergence"))

(validation_predictions
    .filter(col("prediction").isNotNull())
    .select(
        col("user_id").cast("long"),
        col("movie_id").cast("long"),
        col("rating").cast("double"),
        col("prediction").cast("double"),
    )
    .withColumn("implementation", lit("spark"))
    .write.mode("overwrite")
    .option("replaceWhere", "implementation = 'spark'")
    .saveAsTable(f"{tbl}.als_validation"))

In [0]:

print(f"Total error = {E_cur}.  Loop count = {loop_count}")

print(
    f"Mean rating training data = "
    f"{rating_100_users_spark_train.select(mean('rating')).first()[0]:.4f}"
)

# Training predictions
training_predictions = (
    rating_100_users_spark_train
    .join(
        P.withColumnRenamed("factors", "user_factors"),
        on="user_id"
    )
    .join(
        Q.withColumnRenamed("factors", "movie_factors"),
        on="movie_id"
    )
    .withColumn(
        "prediction",
        expr("""
            aggregate(
                zip_with(user_factors, movie_factors, (p, q) -> p * q),
                cast(0.0 as double),
                (acc, x) -> acc + x
            )
        """)
    )
)

# Calculate training RMSE
rmse = training_predictions.select(
    sqrt(mean(pow(col("rating") - col("prediction"), 2))).alias("rmse")
).first()["rmse"]

print(f"Training data RMSE = {rmse:.4f}")

# Maximum factor-vector norms
user_norm_max = P.select(
    spark_max(
        expr("sqrt(aggregate(factors, cast(0.0 as double), (acc, x) -> acc + x*x))")
    ).alias("max_norm")
).first()["max_norm"]

movie_norm_max = Q.select(
    spark_max(
        expr("sqrt(aggregate(factors, cast(0.0 as double), (acc, x) -> acc + x*x))")
    ).alias("max_norm")
).first()["max_norm"]

print(f"||P|| max = {user_norm_max:.4f}")
print(f"||Q|| max = {movie_norm_max:.4f}")
